# MAHAVIR TRUST (MECT) — HIm Intelligent AI Assistant Fine-Tuning
### Model: Qwen/Qwen2.5-3B-Instruct
### Framework: Unsloth QLoRA 4-bit
### Target Output: `qwen2.5-3b-mect-q4_k_m.gguf` for Hostinger KVM 4 VPS CPU Deployment

> **Hardware Requirement:** Free Google Colab with 1x NVIDIA Tesla T4 GPU (Runtime > Change runtime type > T4 GPU).

## 1. Install Unsloth & PEFT Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install datasets transformers huggingface_hub

## 2. Load 4-bit Base Model (`Qwen2.5-3B-Instruct`)

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None  # None for auto detection (Float16 or Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("Base model loaded successfully in 4-bit VRAM!")

## 3. Attach QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("QLoRA adapters successfully configured!")

## 4. Upload MECT Institutional Datasets
Upload `mect_him_train.jsonl` and `mect_him_val.jsonl` using the Colab file uploader or direct curl.

In [ ]:
from google.colab import files
import os

if not os.path.exists("mect_him_train.jsonl"):
    print("Please upload mect_him_train.jsonl and mect_him_val.jsonl:")
    uploaded = files.upload()
else:
    print("Datasets already present!")

## 5. Prepare Dataset with Qwen ChatML Template

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset_train = load_dataset("json", data_files={"train": "mect_him_train.jsonl"})["train"]
dataset_train = dataset_train.map(formatting_prompts_func, batched=True)
print(f"Training examples prepared: {len(dataset_train)}")

## 6. Execute QLoRA Training (~25-35 minutes on T4 GPU)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"Fine-tuning completed! Final loss: {trainer_stats.training_loss:.4f}")

## 7. Trilingual Verification Generation Test

In [ ]:
FastLanguageModel.for_inference(model)

test_queries = [
    ("English", "Can a student from Koraput receive scholarship money in cash?"),
    ("Odia", "ନମସ୍କାର, ମହାବୀର ଟ୍ରଷ୍ଟ କଲେଜ ଫିସ୍ କିପରି ପ୍ରଦାନ କରେ?"),
    ("Hindi", "नमस्ते, क्या मुझे छात्रवृत्ति का पैसा सीधे मेरे बैंक खाते में मिल सकता है?")
]

for lang, q in test_queries:
    messages = [
        {"role": "system", "content": "You are HIm, official digital assistant of Mahavir Trust (MECT)."},
        {"role": "user", "content": q}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=150, use_cache=True, temperature=0.3)
    resp = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"\n--- [{lang}] ---\nQ: {q}\nA: {resp.strip()}")

## 8. Export Model to GGUF `Q4_K_M` for Hostinger VPS Deployment

In [ ]:
# Save directly in 4-bit GGUF format
model.save_pretrained_gguf("mect_him_qwen2.5_3b", tokenizer, quantization_method="q4_k_m")
print("Model converted to GGUF Q4_K_M! Ready to download.")

## 9. Download GGUF Weight File

In [ ]:
# Download the quantized .gguf model to local computer
from google.colab import files
import glob

gguf_files = glob.glob("mect_him_qwen2.5_3b/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("No GGUF file found in output directory.")